# Environment

In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
from datasets import Dataset

## Load Model

In [3]:
model = BertForSequenceClassification.from_pretrained(f"./models/BERTimbau_large_GoEmotions_portuguese")
tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-large-portuguese-cased")

## Pipeline

In [4]:
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=256,
    top_k=None,
    return_all_scores=True,
    device=0
)

Device set to use cuda:0


### Map Symptom

In [5]:
polarity_mapping = {
    -1: {
        "aborrecimento", "constrangimento", "decepção", "desaprovação", "luto",
        "medo", "nervosismo", "nojo", "raiva", "remorso", "tristeza"
    },
    0: {
        "neutro", "curiosidade", "confusão", "percepção", "surpresa"
    },
    1: {
        "admiração", "alegria", "alívio", "aprovação", "amor", "desejo",
        "diversão", "entusiasmo", "gratidão", "orgulho", "otimismo", "zelo"
    }
}

## Extract

In [6]:
amive = pd.read_csv("./data/processed/amive.csv")
dataset = Dataset.from_pandas(amive[["TEXT"]])

In [7]:
results = classifier(list(dataset["TEXT"]))

### Polarity

In [8]:
def map_emotions_to_polarity(emotions):
    polarities = []
    for emo in emotions:
        label = emo["label"].lower()
        score = emo["score"]
        if score >= 0.3:
            for polarity, emotion_set in polarity_mapping.items():
                if label in emotion_set:
                    polarities.append(polarity)
                    break

    total = sum(polarities)
    if total > 0: return 1
    if total < 0: return -1
    return 0

In [9]:
polarities = [map_emotions_to_polarity(r) for r in results]

### All emotions

In [10]:
emotions_df = pd.DataFrame([
    {e["label"].lower(): e["score"] for e in r} for r in results
])

### Save

In [11]:
amive_ge = pd.concat([amive[["DOCNO"]], emotions_df], axis=1)
amive_gep = amive[["DOCNO"]].assign(polarity=polarities)

In [12]:
amive_ge.to_csv("./data/features/amive_ge.csv", index=False)
amive_gep.to_csv("./data/features/amive_gep.csv", index=False)